# EXP 10 — one-click Kaggle run (preprocess, baseline, fine-tune, evaluate)

Import this notebook into Kaggle, set Accelerator = GPU and Internet = On, then use **Save Version → Save & Run All** so the ~1.5 h run continues on Kaggle's servers even if the browser is closed. Outputs: BLEU/chrF table and sample translations printed at the end.

In [ ]:
# One-click pipeline for Kaggle (Settings: Accelerator = GPU, Internet = On), then "Save Version" -> "Save & Run All".
!pip -q install -U transformers datasets evaluate sentencepiece sacrebleu accelerate 2>&1 | tail -2
!git clone -q https://github.com/Kartiken07/LOW-RESOURCE-NEURAL-MACHINE-TRANSLATION.git repo
%cd repo
# 1. data: download Samanantar en-as, clean, 80:10:10 split (seed 42), 40k random pairs
!python src/preprocess.py --hf-dataset samanantar --max-rows 40000 --out-dir data/processed 2>&1 | head -12
# 2. baseline: pretrained NLLB before fine-tuning (same 1000 test pairs as below)
!CUDA_VISIBLE_DEVICES=0 python src/evaluate.py --model facebook/nllb-200-distilled-600M --label "Pretrained (before fine-tuning)" --max-samples 1000 2>&1 | tr '\r' '\n' | grep -E "BLEU|chrF"
# 3. fine-tune on one GPU (2 epochs, batch 4 x grad-accum 4, lr 2e-5, max length 128, seed 42)
!CUDA_VISIBLE_DEVICES=0 python src/train.py --data-dir data/processed --output-dir nllb-en-as --epochs 2 --batch-size 4 --grad-accum 4 --lr 2e-5 --max-length 128 --seed 42 2>&1 | tr '\r' '\n' | grep -E "eval_loss|train_runtime|Training time|train=|Error"
# 4. evaluate the fine-tuned model on the same 1000 test pairs
!CUDA_VISIBLE_DEVICES=0 python src/evaluate.py --model nllb-en-as --label Fine-tuned --max-samples 1000 2>&1 | tr '\r' '\n' | grep -E "BLEU|chrF"
# 5. results
!cat results/metrics.csv
!cat nllb-en-as/training_time_minutes.txt
!rm -rf nllb-en-as/checkpoint-*
!python -c "import pandas as pd; d = pd.read_csv('results/sample_translations.csv'); [print(r.model, '|', r.source, '|', r.reference, '|', r.model_output) for r in d.itertuples()]"
